# ARBORMAP: treino da UrbanHeatNet, simulação em Teresina e exportação do mapa

Este notebook refaz, com o modelo final (UrbanHeatNet, 4 canais: NDVI, NDBI, DEM e NDWI), os resultados que entram no artigo e os arquivos do mapa interativo:

1. treino nos cinco folds da Tabela 1 (com retomada automática se a sessão cair);
2. Tabela 2 e Figuras 2 e 3, geradas do histórico salvo;
3. simulação em Teresina com o modelo do fold em que Teresina é validação (Figuras 4, 5 e 6);
4. exportação georreferenciada para o mapa (`docs/data/`).

**Ambiente:** Ambiente de execução → Alterar tipo → GPU (A100 se disponível: cerca de 75 min no total; T4: cerca de 10 a 11 h).

**Se a sessão cair:** reconecte e rode de novo, de cima para baixo. Folds concluídos são pulados e o fold interrompido continua da última época salva.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess
REPO = '/content/arbormap'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/abraaolm-maker/arbormap.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rasterio'], check=True)
sys.path.insert(0, f'{REPO}/src')

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NENHUMA (troque o ambiente para T4)')

## 1. Caminhos

O dataset é copiado do Drive para o disco local do Colab (leitura muito mais rápida). Os resultados vão para o Drive, para sobreviver a quedas da sessão.

In [ ]:
import shutil
from pathlib import Path

DATA_DRIVE = Path('/content/drive/MyDrive/heat_dataset_v3')    # ajuste se o dataset estiver em outra pasta
DATA_LOCAL = Path('/content/heat_dataset_v3')
OUT = Path('/content/drive/MyDrive/arbormap_final')             # checkpoints, logs, tabelas e figuras

if not DATA_LOCAL.exists():
    shutil.copytree(DATA_DRIVE, DATA_LOCAL, ignore=shutil.ignore_patterns('*.json', '*_poluidas'))
print(len(list(DATA_LOCAL.rglob('*.npy'))), 'arquivos .npy em', DATA_LOCAL)
OUT.mkdir(parents=True, exist_ok=True)

## 2. Treino nos cinco folds

Versão oficial final (`CONFIG_FINAL`). Folds da Tabela 1 do artigo. Modelo e perda da célula 32 (`hybrid_pipeline.py`: UrbanHeatNet, SSIM + L1 + hotspot, lote 2, recorte 400 px), com LR 1e-4, weight decay 0,05 e pixels de LST zerada (lacunas de nuvem do download) excluídos do alvo. **20 épocas fixas, modelo da última época**, e a taxa de aprendizado ajustada pela perda de treino: as cidades de validação não influenciam nem a escolha do modelo nem o treino. O que for usado aqui fica gravado em `config.json`.

Tudo o que aparece na tela também é gravado em `treino_log.txt`.

In [ ]:
from arbormap.model.train_urbanheatnet import run_kfold, FOLDS_ARTIGO, CONFIG_FINAL

resultados = run_kfold(DATA_LOCAL, OUT, folds=FOLDS_ARTIGO, config=CONFIG_FINAL)

## 3. Tabela 2 e Figuras 2 e 3

`tabela2.csv` traz as métricas da época 20 de cada fold, com média ± desvio-padrão amostral. As figuras usam o fold escolhido em `FOLD_FIGURAS` (0 a 4). `curvas_todos_folds.png` mostra treino e validação por época nos cinco folds.

In [ ]:
import json
from IPython.display import Image, display
from arbormap.analysis.figures_urbanheatnet import load_fold, figure_training_curves, figure_std

print((OUT / 'tabela2.csv').read_text(encoding='utf-8-sig'))

FOLD_FIGURAS = 1   # índice 0-4; o artigo atual usa o Fold 2 (índice 1)
FIG = OUT / 'figuras'; FIG.mkdir(exist_ok=True)
fr = load_fold(OUT, FOLD_FIGURAS)
figure_training_curves(fr, FIG / 'Figura2_Curvas_treinamento.png')
figure_std(fr, FIG / 'Figura3_Desvio_padrao.png')
display(Image(str(FIG / 'Figura2_Curvas_treinamento.png'), width=900))
display(Image(str(FIG / 'Figura3_Desvio_padrao.png'), width=600))

from arbormap.analysis.verify_urbanheatnet import plot_all_folds
plot_all_folds({'final': OUT}, FIG / 'curvas_todos_folds.png')
display(Image(str(FIG / 'curvas_todos_folds.png'), width=1400))

## 4. Simulação em Teresina (Figuras 4, 5 e 6)

Usa o modelo do fold em que Teresina é **cidade de validação**, isto é, um modelo que nunca viu Teresina no treino (com os folds da Tabela 1, é o Fold 1, arquivo `best_hybrid_fold_0.pt`, numerado a partir de 0). A célula confere isso pelas cidades gravadas no próprio checkpoint.

A cena é escolhida começando pelo par mais recente e voltando até achar um em que as duas cenas estejam entre as 30% mais limpas da cidade (escore de nuvem = lacunas zeradas + manchas frias em terra firme). A prancha `cenas_recentes.png` mostra as cenas consideradas; para usar outra, troque `PAR_CENA` pelo índice desejado.

In [ ]:
import torch
from arbormap.model.hybrid_pipeline import filter_cloudy_files_runtime
from arbormap.decision import simulate_urbanheatnet as sim

CIDADE, ORCAMENTO_FIGURA = 'Teresina', 45
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

res_kfold = json.loads((OUT / 'resultados_kfold.json').read_text(encoding='utf-8'))
fold_t = sim.fold_of_city(res_kfold['folds'], CIDADE)
hs_ref = res_kfold['folds'][fold_t]['best']['hotspot_mae_c']
print(f'{CIDADE} é validação no Fold {fold_t + 1}; Hotspot MAE desse fold = {hs_ref:.2f} °C')

model, state = sim.load_fold_model(OUT / f'best_hybrid_fold_{fold_t}.pt', device)
assert CIDADE in state['val_cities'] and CIDADE not in state['train_cities'], 'o modelo viu a cidade no treino'
print('Modelo usado:', f'best_hybrid_fold_{fold_t}.pt', '| validação:', state['val_cities'], '| época:', state['epoch'])
valid_files = filter_cloudy_files_runtime(list((DATA_LOCAL / CIDADE).glob('*.npy')))
PAR_CENA, tabela_cenas = sim.select_recent_clear_pair(DATA_LOCAL, CIDADE, valid_files)
sim.figure_scene_sheet(DATA_LOCAL, CIDADE, tabela_cenas, FIG / 'cenas_recentes.png')
display(Image(str(FIG / 'cenas_recentes.png'), width=1100))
print('Cena escolhida:', tabela_cenas[PAR_CENA])
res = sim.run_city(model, state, DATA_LOCAL, valid_files, city=CIDADE, pair_index=PAR_CENA,
                   budget_figura=ORCAMENTO_FIGURA, hotspot_mae_ref=hs_ref)

mae_cena = sim.figure_obs_pred_error(res, FIG / 'Figura4_LST_observado_previsto_erro.png')
sim.figure_strategies(res, FIG / 'Figura5_Resfriamento_por_estrategia.png', hotspot_label=f'Hotspot MAE (Fold {fold_t + 1})')
sim.figure_budget_curve(res, FIG / 'Figura6_Eficiencia_vs_orcamento.png')
sim.save_summary(res, OUT)
print('MAE na cena exibida:', round(mae_cena, 2), '°C | cena:', res['meta']['arquivo_base'], '→', res['meta']['arquivo_alvo'])
for f in ['Figura4_LST_observado_previsto_erro.png', 'Figura5_Resfriamento_por_estrategia.png', 'Figura6_Eficiencia_vs_orcamento.png']:
    display(Image(str(FIG / f), width=900))

## 5. Arquivos do mapa interativo

Com `USAR_EE = True`, as bordas da imagem vêm do Earth Engine (o mesmo retângulo do download; pede autenticação). Com `False`, usa-se a aproximação geodésica, com erro da ordem de um pixel. Os arquivos ficam em `OUT/mapa` no Drive; com `BAIXAR_ZIP = True`, o navegador também baixa um `.zip`.

In [ ]:
from arbormap.decision.export_map import export_city, exact_bounds_ee, CIDADES_LONLAT

USAR_EE = False
full_bounds = None
if USAR_EE:
    import ee
    ee.Authenticate(); ee.Initialize(project='script-39a43')
    full_bounds = exact_bounds_ee(*CIDADES_LONLAT[CIDADE])

MAPA = OUT / 'mapa'
info = export_city(res, MAPA, prefix=CIDADE.lower(), full_bounds=full_bounds)
print(json.dumps({k: info[k] for k in ('bounds_leaflet', 'fonte_bounds', 'legenda_lst_c', 'arquivos_tif_gravados')}, indent=2, ensure_ascii=False))

BAIXAR_ZIP = False
if BAIXAR_ZIP:
    shutil.make_archive('/content/arbormap_mapa', 'zip', MAPA)
    from google.colab import files
    files.download('/content/arbormap_mapa.zip')

## 6. Verificação independente

Mede, nos mesmos pixels limpos, o erro do modelo e de referências simples: prever a média da cena, regressão linear com os mesmos 4 índices e persistência (LST do mês anterior, que usa LST e serve só de comparação). Também confere que nenhum canal de entrada é a LST. Resultado em `OUT/verificacao/`.

In [ ]:
from arbormap.analysis.verify_urbanheatnet import run_verification
resumo = run_verification({'final': OUT}, DATA_LOCAL, OUT / 'verificacao')

## O que levar para o artigo

- `tabela2.csv` e `resultados_kfold.json`: Tabela 2 nova (os valores mudam em relação à versão atual).
- `config.json`: hiperparâmetros a declarar na Seção III-D; a perda é SSIM + L1 + hotspot (Seções III-B e III-E).
- `figuras/`: Figuras 2 a 6 regeneradas.
- `teresina_simulacao.json`: redução por estratégia e orçamento (a aleatória com média ± DP de 20 sorteios), fold e cena usados.
- `treino_log.txt`: log completo, para o diário de bordo.